# GNN Cloud Notebook: GCN-3 Residual Huber Multi-Seed

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and validates the Huber-loss residual GCN setup across multiple seeds.

This notebook keeps the following improvements fixed:
- Adam with the two-phase schedule
- weighted edges from `adjacency_area.csv`
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling
- standardized regression targets during training
- residual/skip-connected 3-layer GCN
- narrower residual learning-rate schedule that improved multi-seed stability

Objective setting validated in this notebook:
- replace MSE loss with Huber/SmoothL1 loss
- keep the stabilized residual architecture and optimizer schedule fixed
- test whether Huber improves seed-tail stability enough to justify promotion or continued use

No new lattice sets are required.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')
        


In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')
        


In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_residual_huber_multi_seed'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/gcn3_residual_huber_multi_seed'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
gnn_root = pipeline_root / 'gnn_prototype'
candidate_module_dirs = [
    gnn_root,
    gnn_root / 'GCN_Optimization',
]
module_dir = None
for candidate in candidate_module_dirs:
    if (candidate / 'colab_gnn_stiffness_prototype.py').is_file():
        module_dir = candidate
        break
if module_dir is None:
    raise FileNotFoundError(f'Could not locate colab_gnn_stiffness_prototype.py under {gnn_root}')

if (gnn_root / 'GCN_Optimization' / 'architecture_comparison_runner.py').is_file():
    optimization_dir = gnn_root / 'GCN_Optimization'
elif (module_dir / 'architecture_comparison_runner.py').is_file():
    optimization_dir = module_dir
else:
    raise FileNotFoundError(f'Could not locate architecture_comparison_runner.py under {gnn_root}')

os.chdir(pipeline_root)

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/gcn3_residual_huber_multi_seed') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs' / 'gcn3_residual_huber_multi_seed'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
per_model_output_root = output_dir / 'per_model'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'GNN root: {gnn_root}')
print(f'Module dir: {module_dir}')
print(f'Optimization dir: {optimization_dir}')
print(f'Working directory: {Path.cwd()}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')
        


In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from architecture_comparison_runner import ArchitectureConfig, run_architecture_experiment


def build_aggregate_frame(summary_frame: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        'Validation_R2',
        'Test_R2',
        'Test_RMSE',
        'Prediction_R2',
        'Prediction_RMSE',
    ]
    aggregate = summary_frame[metric_columns].agg(['mean', 'std', 'min', 'max']).T.reset_index()
    aggregate.columns = ['Metric', 'mean', 'std', 'min', 'max']
    return aggregate


def plot_multi_seed_summary(summary_frame: pd.DataFrame, save_path: Path) -> None:
    figure, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()
    metrics = [
        ('Test_R2', 'Test R2'),
        ('Test_RMSE', 'Test RMSE'),
        ('Prediction_R2', 'Prediction R2'),
        ('Validation_R2', 'Validation R2'),
    ]

    for axis, (metric_key, title) in zip(axes, metrics):
        axis.plot(summary_frame['Seed'], summary_frame[metric_key], marker='o', linewidth=2, color='#b56576')
        axis.set_title(title)
        axis.set_xlabel('Seed')
        axis.grid(alpha=0.3)

    figure.tight_layout()
    figure.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close(figure)
        


In [ ]:
SEEDS = (11, 42, 73, 101, 202)
HIDDEN_DIM = 24
DROPOUT = 0.10
WEIGHT_DECAY = 1e-5
LR_PHASE1 = 0.0025
LR_PHASE2 = 0.00045
LOSS_NAME = 'huber'
HUBER_BETA = 0.75
ARCHITECTURE_NAME = 'gcn3_residual'
ARCHITECTURE_LABEL = 'GCN-3 Residual Huber'

print(f'Seeds: {SEEDS}')
print(f'Hidden dim: {HIDDEN_DIM}')
print(f'Dropout: {DROPOUT}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'LR phase 1: {LR_PHASE1}')
print(f'LR phase 2: {LR_PHASE2}')
print(f'Loss: {LOSS_NAME} (beta={HUBER_BETA})')
print(f'Architecture: {ARCHITECTURE_LABEL}')


In [ ]:
experiment_results = {}
summary_rows = []

for seed in SEEDS:
    print(f'=== Running seed {seed} ===')
    config = ArchitectureConfig(
        architecture_name=ARCHITECTURE_NAME,
        architecture_label=ARCHITECTURE_LABEL,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        weight_decay=WEIGHT_DECAY,
        lr_phase1=LR_PHASE1,
        lr_phase2=LR_PHASE2,
        loss_name=LOSS_NAME,
        huber_beta=HUBER_BETA,
        seed=seed,
        output_group='gcn3_residual_huber_multi_seed',
    )
    result = run_architecture_experiment(
        config,
        train_root=train_root,
        predict_root=predict_root,
        output_root=per_model_output_root,
    )
    experiment_results[seed] = result
    row = result['summary_frame'].iloc[0].to_dict()
    row['Output_Dir'] = str(result['output_dir'])
    summary_rows.append(row)

summary_frame = pd.DataFrame(summary_rows).sort_values('Seed').reset_index(drop=True)
summary_path = output_dir / 'gcn3_residual_huber_multi_seed_summary.csv'
summary_frame.to_csv(summary_path, index=False)

print(f'Saved per-seed summary to {summary_path}')
display(summary_frame)


In [ ]:
aggregate_frame = build_aggregate_frame(summary_frame)
aggregate_path = output_dir / 'gcn3_residual_huber_multi_seed_aggregate.csv'
aggregate_frame.to_csv(aggregate_path, index=False)

summary_plot_path = output_dir / 'gcn3_residual_huber_multi_seed_metric_summary.png'
plot_multi_seed_summary(summary_frame, save_path=summary_plot_path)

best_seed_row = summary_frame.sort_values('Test_R2', ascending=False).iloc[0]
best_seed_payload = best_seed_row.to_dict()
best_seed_path = output_dir / 'gcn3_residual_huber_best_seed_summary.json'
best_seed_path.write_text(json.dumps(best_seed_payload, indent=2), encoding='utf-8')

run_metadata = {
    'architecture': ARCHITECTURE_LABEL,
    'architecture_key': ARCHITECTURE_NAME,
    'seeds': list(SEEDS),
    'hidden_dim': HIDDEN_DIM,
    'dropout': DROPOUT,
    'weight_decay': WEIGHT_DECAY,
    'lr_phase1': LR_PHASE1,
    'lr_phase2': LR_PHASE2,
    'loss_name': LOSS_NAME,
    'huber_beta': HUBER_BETA,
    'best_seed': int(best_seed_row['Seed']),
    'best_test_r2': float(best_seed_row['Test_R2']),
    'mean_test_r2': float(aggregate_frame.loc[aggregate_frame['Metric'] == 'Test_R2', 'mean'].iloc[0]),
    'mean_prediction_r2': float(aggregate_frame.loc[aggregate_frame['Metric'] == 'Prediction_R2', 'mean'].iloc[0]),
}
metadata_path = output_dir / 'gcn3_residual_huber_multi_seed_run_metadata.json'
metadata_path.write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

print(f'Saved aggregate summary to {aggregate_path}')
print(f'Saved summary plot to {summary_plot_path}')
print(f'Saved best-seed summary to {best_seed_path}')
print(f'Saved run metadata to {metadata_path}')
display(aggregate_frame)
print('Best seed by Test R2:')
display(summary_frame[summary_frame['Seed'] == int(best_seed_row['Seed'])])


In [ ]:
print(f'Parent run dir: {output_dir}')
parent_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('Parent-level files:')
for file_name in parent_files:
    print(f' - {file_name}')

for seed, result in experiment_results.items():
    print(f' - seed {seed}: {result["output_dir"]}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODEL_TO_GITHUB:
        for model_path in git_run_dir.rglob('lattice_gnn_model.pt'):
            model_path.unlink()

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add narrow-LR retuned GCN-3 residual multi-seed cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)
        


In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_gcn3_residual_huber_multi_seed.zip'
    !cd /content && zip -qr gnn_outputs_gcn3_residual_huber_multi_seed.zip gnn_outputs/gcn3_residual_huber_multi_seed
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')
        
